using `Qwen3.5:4b`, we generate Q&A style training data, giving the model the wikipedia passage as context. we take a random sample of 25,000 articles, and generate a single Q&A style training point from each of them, amounting to 25,000 rows of training data for SFT. we do *not* allow the model to think; mostly due to throughput.

In [1]:
import os
import duckdb
import ollama
import pandas as pd

from dotenv import load_dotenv
from concurrent.futures import ThreadPoolExecutor

load_dotenv()

DATA_PATH = os.environ['DATAPATH']

we use `duckdb` to sample from the parquet file, as the file is large.

In [2]:
tr_data = duckdb.sql(f"select text from '{DATA_PATH}/train.parquet' using sample 25000 rows").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

training samples, after processing, will aim to looke like `User: 'insert question here' Assistant: 'insert response here'`. to assure that the model will produce training samples that align with this strict format, we can use structured outputs and pass in a schema, using `pydantic`.

In [3]:
from pydantic import BaseModel
from typing import List

class Sample(BaseModel):
    user: str
    assistant: str

now generate the training samples. the qwen model is guided to create succinct, and informative q&a training points, with an emphasis on pulling information from the data. `ollama` guarantees that the model will return a message in the format as specified by the model class `Sample`.

In [ ]:
MAX_PAGE_CHARS = 600

def truncate_page(page: str, max_chars: int = MAX_PAGE_CHARS) -> str:
    if len(page) <= max_chars:
        return page
    cut = page[:max_chars]
    end = max(cut.rfind('\n\n'), cut.rfind('. '))
    return cut[:end + 1] if end > max_chars // 2 else cut

def create_prompt(page: str) -> str:
    return f"""
    <page data>
    {truncate_page(page)}
    </page data>

    <instruction>
    You are tasked with creating training samples for post-training an LLM to respond to Q&A style prompts.
    In <page data>, there is cleaned text data from a single Wikipedia page. Your goal is to produce a training data point
    in the format specified, which captures a simple question, and a **correct** answer, drawing from the data.
    Responses should be fact-of-the-matter, and verbatim from the data, where possible. Responses should be informative, and elaborate if necessary.
    </instruction>
    """

def create_data_point(page: str) -> str:
    return Sample.model_validate_json(
        ollama.chat(
            model='qwen3.5:4b', 
            messages=[{'role': 'user', 'content': create_prompt(page)}], 
            think=False, 
            format=Sample.model_json_schema()
        ).message.content
    )

In [ ]:
import json
from concurrent.futures import ThreadPoolExecutor

sft_bulk = []
WORKERS = 8
OUT_PATH = f'{DATA_PATH}/sft_tr.jsonl'

def safe_create(text):
    try:
        return create_data_point(text)
    except Exception as e:
        print(e)
        return None

with ThreadPoolExecutor(max_workers=WORKERS) as executor, open(OUT_PATH, 'w', encoding='utf-8') as f:
    for i, data_point in enumerate(executor.map(safe_create, tr_data["text"])):
        if i % 10 == 0: print(f'progress {(i / len(tr_data)) * 100:.4f}%')
        if data_point is not None:
            row = data_point.model_dump()
            sft_bulk.append(row)
            f.write(json.dumps(row, ensure_ascii=False) + '\n')
            f.flush()

print(f'saved {len(sft_bulk)} samples to {OUT_PATH}')

progress 0.0000%
progress 0.0400%
progress 0.0800%
progress 0.1200%
progress 0.1600%
progress 0.2000%
progress 0.2400%
progress 0.2800%
progress 0.3200%
progress 0.3600%
progress 0.4000%
progress 0.4400%
progress 0.4800%
progress 0.5200%
progress 0.5600%
progress 0.6000%
progress 0.6400%
progress 0.6800%
progress 0.7200%
progress 0.7600%
progress 0.8000%
progress 0.8400%


we can now merge the strings in the pre-specified format, and then tokenise the training samples.